# Phase 4: residual and MC-Dropout reliability

Validate whether prediction residual and LSTM epistemic uncertainty provide genuinely different diagnostic information before either signal is used by a controller.

In [1]:
import json
from dataclasses import replace
from pathlib import Path
import sys
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import torch

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from motor_model import DCMotorParams, simulate_motor
from reliability import (
    DIAGNOSTIC_STATE_NAMES,
    calibrate_thresholds,
    classify_diagnostics,
    load_lstm_model,
    mc_dropout_predict,
)

## Load the Phase 2–3 artifacts

In [2]:
SEED = 2026
FAULT_START = 4.0
MC_BATCH_SIZE = 256
MC_PASS_VALUES = (10, 20, 30, 50)
PERCENTILES = (90.0, 95.0, 99.0)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
dataset_path = project_root / 'data' / 'processed' / 'dc_motor_lstm_dataset.npz'
weights_path = project_root / 'results' / 'lstm_model_weights.pt'
model_config_path = project_root / 'results' / 'configs' / 'lstm_model_config.json'

data = np.load(dataset_path)
model, model_config = load_lstm_model(weights_path, model_config_path)
normalization = model_config['normalization']
input_mean = np.asarray(normalization['input_mean'])
input_std = np.asarray(normalization['input_std'])
target_mean = np.asarray(normalization['target_mean'])
target_std = np.asarray(normalization['target_std'])
window_length = model_config['window_length']

assert np.allclose(input_mean, data['normalization_input_mean'])
assert np.allclose(target_std, data['normalization_target_std'])
print(f'Loaded model; window={window_length}, recurrent dropout={model_config["model"]["dropout"]}')

Loaded model; window=20, recurrent dropout=0.2


## Clean validation calibration data

Thresholds are derived only from the clean validation trajectories. No test scenario is used to fit either threshold.

In [3]:
def align_split_field(split_name, field):
    run_ids = data[f'window_run_ids_{split_name}']
    aligned = np.empty(len(run_ids), dtype=np.float32)
    for run_id in np.unique(run_ids):
        positions = np.flatnonzero(run_ids == run_id)
        aligned[positions] = data[field][
            run_id, window_length : window_length + len(positions)
        ]
    return aligned


def mc_physical(inputs, passes, seed):
    torch.manual_seed(seed)
    mean, variance = mc_dropout_predict(
        model, inputs, passes=passes, batch_size=MC_BATCH_SIZE
    )
    mean = mean.numpy()[:, 0] * target_std[0] + target_mean[0]
    variance = variance.numpy()[:, 0] * target_std[0] ** 2
    return mean, variance


validation_measured = align_split_field('validation', 'y_measured')
X_validation = data['X_validation']

## Controlled test scenarios

Sensor faults are applied only at evaluation time. The load and parameter-shift cases are newly simulated and move outside the Phase 2 training range after 4 seconds.

In [4]:
nominal_values = data['parameter_values'][0]
nominal_run_ids = np.flatnonzero(
    np.all(np.isclose(data['parameter_values'], nominal_values, rtol=0.0, atol=1e-8), axis=1)
)
base_run = next(
    int(run_id) for run_id in data['split_run_ids_test'] if run_id in nominal_run_ids
)
base_time = data['time'].astype(float)
base_voltage = data['voltage'][base_run].astype(float)
base_true = data['y_true'][base_run].astype(float)
base_measured = data['y_measured'][base_run].astype(float)
base_load = data['load_torque'][base_run].astype(float)
fault_mask = base_time >= FAULT_START
scenario_rng = np.random.default_rng(SEED + 40)

bias_measurement = base_measured.copy()
bias_measurement[fault_mask] += 6.0
dropout_measurement = base_measured.copy()
dropout_measurement[fault_mask] = 0.0
noisy_measurement = base_measured.copy()
noisy_measurement[fault_mask] += scenario_rng.normal(0.0, 2.0, fault_mask.sum())

scenarios = [
    {
        'name': 'clean', 'title': 'A. Clean operation', 'time': base_time,
        'voltage': base_voltage, 'true': base_true, 'measured': base_measured,
        'load': base_load, 'fault_start': None,
    },
    {
        'name': 'sensor_bias', 'title': 'B. +6 rad/s sensor bias', 'time': base_time,
        'voltage': base_voltage, 'true': base_true, 'measured': bias_measurement,
        'load': base_load, 'fault_start': FAULT_START,
    },
    {
        'name': 'sensor_dropout', 'title': 'C. Sensor dropout to zero', 'time': base_time,
        'voltage': base_voltage, 'true': base_true, 'measured': dropout_measurement,
        'load': base_load, 'fault_start': FAULT_START,
    },
    {
        'name': 'increased_noise', 'title': 'D. Added 2 rad/s sensor noise', 'time': base_time,
        'voltage': base_voltage, 'true': base_true, 'measured': noisy_measurement,
        'load': base_load, 'fault_start': FAULT_START,
    },
]

In [5]:
nominal_params = DCMotorParams()

def scenario_voltage(time):
    value = 6.0 + 3.0 * np.sin(2 * np.pi * 0.23 * time) + 2.0 * np.sin(2 * np.pi * 0.71 * time)
    return float(np.clip(value, 0.0, 12.0))


outside_load = lambda time: 0.03 if time < FAULT_START else 0.15
outside_time, outside_states = simulate_motor(
    scenario_voltage,
    outside_load,
    simulation_time=float(data['duration']),
    timestep=float(data['timestep']),
    params=nominal_params,
)
outside_true = outside_states[:, 1]
outside_measured = outside_true + scenario_rng.normal(0.0, 0.25, len(outside_true))
scenarios.append(
    {
        'name': 'outside_load', 'title': 'E. Load step to 0.15 N m', 'time': outside_time,
        'voltage': np.array([scenario_voltage(t) for t in outside_time]),
        'true': outside_true, 'measured': outside_measured,
        'load': np.array([outside_load(t) for t in outside_time]),
        'fault_start': FAULT_START,
    }
)

In [6]:
shifted_params = replace(
    nominal_params,
    resistance=1.20 * nominal_params.resistance,
    inductance=0.85 * nominal_params.inductance,
    back_emf_constant=1.15 * nominal_params.back_emf_constant,
    torque_constant=0.85 * nominal_params.torque_constant,
    inertia=1.20 * nominal_params.inertia,
    viscous_friction=1.20 * nominal_params.viscous_friction,
    coulomb_friction=1.20 * nominal_params.coulomb_friction,
)
constant_load = lambda _time: 0.03
pre_time, pre_states = simulate_motor(
    scenario_voltage, constant_load, simulation_time=FAULT_START,
    timestep=float(data['timestep']), params=nominal_params
)
post_time, post_states = simulate_motor(
    lambda time: scenario_voltage(time + FAULT_START),
    constant_load,
    simulation_time=float(data['duration']) - FAULT_START,
    timestep=float(data['timestep']),
    params=shifted_params,
    initial_state=tuple(pre_states[-1]),
)
shift_time = np.concatenate((pre_time, FAULT_START + post_time[1:]))
shift_states = np.vstack((pre_states, post_states[1:]))
shift_true = shift_states[:, 1]
shift_measured = shift_true + scenario_rng.normal(0.0, 0.25, len(shift_true))
scenarios.append(
    {
        'name': 'parameter_shift', 'title': 'F. Sudden 15–20% parameter shift',
        'time': shift_time,
        'voltage': np.array([scenario_voltage(t) for t in shift_time]),
        'true': shift_true, 'measured': shift_measured,
        'load': np.full(len(shift_time), 0.03), 'fault_start': FAULT_START,
    }
)
print('Scenarios:', [scenario['name'] for scenario in scenarios])

Scenarios:

 ['clean', 'sensor_bias', 'sensor_dropout', 'increased_noise', 'outside_load', 'parameter_shift']


In [7]:
def normalized_scenario_windows(scenario):
    features = np.column_stack((scenario['voltage'], scenario['measured']))
    windows = np.array(
        [features[start : start + window_length] for start in range(len(features) - window_length)],
        dtype=np.float32,
    )
    return ((windows - input_mean) / input_std).astype(np.float32)


scenario_inputs = {
    scenario['name']: normalized_scenario_windows(scenario) for scenario in scenarios
}

## MC-pass and percentile sensitivity

For each MC count, thresholds are recalibrated from the same clean validation split. The final percentile is selected only to keep clean-validation combined false alarms below 5%; fault cases do not tune the thresholds.

In [8]:
prediction_cache = {}
sensitivity_rows = []

for passes in MC_PASS_VALUES:
    started = perf_counter()
    validation_mean, validation_variance = mc_physical(X_validation, passes, SEED)
    validation_residual = np.abs(validation_measured - validation_mean)
    scenario_statistics = {}
    for scenario_index, scenario in enumerate(scenarios):
        mean, variance = mc_physical(
            scenario_inputs[scenario['name']], passes, SEED + 100 + scenario_index
        )
        scenario_statistics[scenario['name']] = {
            'mean': mean,
            'variance': variance,
            'residual': np.abs(scenario['measured'][window_length:] - mean),
        }
    elapsed = perf_counter() - started
    prediction_cache[passes] = {
        'validation_residual': validation_residual,
        'validation_variance': validation_variance,
        'scenarios': scenario_statistics,
    }

    for percentile in PERCENTILES:
        thresholds = calibrate_thresholds(
            validation_residual, validation_variance, percentile
        )
        validation_codes = classify_diagnostics(
            validation_residual, validation_variance, **thresholds
        )
        detection_rates = []
        sensor_quadrant_rates = []
        model_quadrant_rates = []
        for scenario in scenarios[1:]:
            statistics = scenario_statistics[scenario['name']]
            codes = classify_diagnostics(
                statistics['residual'], statistics['variance'], **thresholds
            )
            target_time = scenario['time'][window_length:]
            post_fault = target_time >= scenario['fault_start']
            detection_rates.append(np.mean(codes[post_fault] != 0))
            if scenario['name'] in ('sensor_bias', 'sensor_dropout', 'increased_noise'):
                sensor_quadrant_rates.append(np.mean(np.isin(codes[post_fault], (1, 3))))
            else:
                model_quadrant_rates.append(np.mean(np.isin(codes[post_fault], (2, 3))))
        sensitivity_rows.append(
            {
                'passes': passes,
                'percentile': percentile,
                'tau_r': thresholds['tau_r'],
                'tau_sigma': thresholds['tau_sigma'],
                'validation_false_alarm_rate': float(np.mean(validation_codes != 0)),
                'mean_fault_detection_rate': float(np.mean(detection_rates)),
                'sensor_quadrant_rate': float(np.mean(sensor_quadrant_rates)),
                'model_quadrant_rate': float(np.mean(model_quadrant_rates)),
                'runtime_seconds': elapsed,
            }
        )

print(' N  pct   val FAR  mean detect  sensor quadrant  model quadrant  runtime(s)')
for row in sensitivity_rows:
    print(
        f"{row['passes']:2d} {row['percentile']:4.0f}  "
        f"{row['validation_false_alarm_rate']:8.3f}  "
        f"{row['mean_fault_detection_rate']:11.3f}  "
        f"{row['sensor_quadrant_rate']:15.3f}  "
        f"{row['model_quadrant_rate']:14.3f}  "
        f"{row['runtime_seconds']:10.2f}"
    )

 N  pct   val FAR  mean detect  sensor quadrant  model quadrant  runtime(s)
10   90     0.189        0.377            0.332           0.019        6.84
10   95     0.097        0.287            0.305           0.005        6.84
10   99     0.020        0.195            0.265           0.000        6.84
20   90     0.189        0.398            0.334           0.014       12.33
20   95     0.098        0.292            0.304           0.004       12.33
20   99     0.020        0.200            0.266           0.001       12.33
30   90     0.189        0.400            0.334           0.009       18.39
30   95     0.099        0.280            0.303           0.004       18.39
30   99     0.020        0.200            0.266           0.000       18.39
50   90     0.189        0.416            0.331           0.016       33.12
50   95     0.098        0.274            0.306           0.004       33.12
50   99     0.020        0.200            0.265           0.000       33.12


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for percentile in PERCENTILES:
    rows = [row for row in sensitivity_rows if row['percentile'] == percentile]
    axes[0].plot(
        [row['passes'] for row in rows],
        [row['validation_false_alarm_rate'] for row in rows],
        marker='o', label=f'{percentile:.0f}th percentile',
    )
    axes[1].plot(
        [row['passes'] for row in rows],
        [row['mean_fault_detection_rate'] for row in rows],
        marker='o', label=f'{percentile:.0f}th percentile',
    )
axes[0].axhline(0.05, color='black', linestyle='--', label='5% target')
axes[0].set_ylabel('Clean-validation false alarm rate')
axes[1].set_ylabel('Mean fault-scenario detection rate')
for axis in axes:
    axis.set_xlabel('MC passes')
    axis.set_xticks(MC_PASS_VALUES)
    axis.grid(alpha=0.3)
    axis.legend()
fig.tight_layout()
plt.show()

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_3376\3379156340.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Final calibration and four-state diagnostics

In [10]:
FINAL_MC_PASSES = 10
eligible_percentiles = [
    row['percentile']
    for row in sensitivity_rows
    if row['passes'] == FINAL_MC_PASSES
    and row['validation_false_alarm_rate'] <= 0.05
]
FINAL_PERCENTILE = min(eligible_percentiles) if eligible_percentiles else max(PERCENTILES)
final_cache = prediction_cache[FINAL_MC_PASSES]
thresholds = calibrate_thresholds(
    final_cache['validation_residual'],
    final_cache['validation_variance'],
    FINAL_PERCENTILE,
)
print(
    f'Chosen N={FINAL_MC_PASSES}, percentile={FINAL_PERCENTILE:.0f}, '
    f'tau_r={thresholds["tau_r"]:.6f} rad/s, '
    f'tau_sigma={thresholds["tau_sigma"]:.8f} (rad/s)^2'
)

Chosen N=10, percentile=99, tau_r=0.724422 rad/s, tau_sigma=0.02396339 (rad/s)^2


In [11]:
def evaluate_scenario(scenario, statistics):
    target_time = scenario['time'][window_length:]
    codes = classify_diagnostics(
        statistics['residual'], statistics['variance'], **thresholds
    )
    flagged = codes != 0
    percentages = {
        name: float(100 * np.mean(codes == code))
        for code, name in enumerate(DIAGNOSTIC_STATE_NAMES)
    }
    if scenario['fault_start'] is None:
        detection_rate = None
        false_alarm_rate = float(np.mean(flagged))
        latency = None
        post_percentages = percentages
    else:
        post_fault = target_time >= scenario['fault_start']
        pre_fault = ~post_fault
        detected_times = target_time[post_fault & flagged]
        detection_rate = float(np.mean(flagged[post_fault]))
        false_alarm_rate = float(np.mean(flagged[pre_fault]))
        latency = (
            float(detected_times[0] - scenario['fault_start'])
            if len(detected_times) else None
        )
        post_percentages = {
            name: float(100 * np.mean(codes[post_fault] == code))
            for code, name in enumerate(DIAGNOSTIC_STATE_NAMES)
        }
    return codes, {
        'fault_detection_rate': detection_rate,
        'false_alarm_rate': false_alarm_rate,
        'detection_latency_seconds': latency,
        'state_percentages_all': percentages,
        'state_percentages_post_fault': post_percentages,
    }


diagnostic_results = {}
scenario_metrics = {}
for scenario in scenarios:
    statistics = final_cache['scenarios'][scenario['name']]
    codes, metrics = evaluate_scenario(scenario, statistics)
    diagnostic_results[scenario['name']] = {**statistics, 'codes': codes}
    scenario_metrics[scenario['name']] = metrics
    print(
        scenario['name'],
        'detection=', metrics['fault_detection_rate'],
        'false_alarm=', round(metrics['false_alarm_rate'], 4),
        'latency=', metrics['detection_latency_seconds'],
        'post states=', {key: round(value, 1) for key, value in metrics['state_percentages_post_fault'].items()},
    )

clean detection= None false_alarm= 0.0135 latency= None post states= {np.str_('NORMAL'): 98.6, np.str_('SENSOR_FAULT'): 1.4, np.str_('MODEL_UNCERTAINTY'): 0.0, np.str_('BOTH'): 0.0}
sensor_bias detection= 0.026217228464419477 false_alarm= 0.0237 latency= 0.0 post states= {np.str_('NORMAL'): 97.4, np.str_('SENSOR_FAULT'): 1.7, np.str_('MODEL_UNCERTAINTY'): 0.5, np.str_('BOTH'): 0.4}
sensor_dropout detection= 0.031210986267166042 false_alarm= 0.0237 latency= 0.0 post states= {np.str_('NORMAL'): 96.9, np.str_('SENSOR_FAULT'): 0.1, np.str_('MODEL_UNCERTAINTY'): 1.1, np.str_('BOTH'): 1.9}
increased_noise detection= 0.8876404494382022 false_alarm= 0.0263 latency= 0.039999961853027344 post states= {np.str_('NORMAL'): 11.2, np.str_('SENSOR_FAULT'): 30.1, np.str_('MODEL_UNCERTAINTY'): 13.5, np.str_('BOTH'): 45.2}
outside_load detection= 0.018726591760299626 false_alarm= 0.0158 latency= 0.040000000000000036 post states= {np.str_('NORMAL'): 98.1, np.str_('SENSOR_FAULT'): 1.9, np.str_('MODEL_UNCER

## Scenario plots

In [12]:
for scenario in scenarios:
    target_time = scenario['time'][window_length:]
    statistics = diagnostic_results[scenario['name']]
    fig, axes = plt.subplots(4, 1, sharex=True, figsize=(11, 9))
    axes[0].plot(target_time, scenario['true'][window_length:], label='true speed')
    axes[0].plot(target_time, scenario['measured'][window_length:], alpha=0.55, label='measured')
    axes[0].plot(target_time, statistics['mean'], '--', label='LSTM MC mean')
    axes[0].set_ylabel('Speed (rad/s)')
    axes[0].legend(ncol=3)
    axes[1].plot(target_time, statistics['residual'])
    axes[1].axhline(thresholds['tau_r'], color='tab:red', linestyle='--', label='tau_r')
    axes[1].set_ylabel('Residual (rad/s)')
    axes[1].legend()
    axes[2].plot(target_time, statistics['variance'])
    axes[2].axhline(
        thresholds['tau_sigma'], color='tab:red', linestyle='--', label='tau_sigma'
    )
    axes[2].set_ylabel('MC variance')
    axes[2].legend()
    axes[3].step(target_time, statistics['codes'], where='post')
    axes[3].set_yticks(range(4), DIAGNOSTIC_STATE_NAMES)
    axes[3].set_ylim(-0.3, 3.3)
    axes[3].set_ylabel('Diagnostic')
    axes[3].set_xlabel('Time (s)')
    if scenario['fault_start'] is not None:
        for axis in axes:
            axis.axvline(scenario['fault_start'], color='black', linestyle=':', alpha=0.7)
    for axis in axes:
        axis.grid(alpha=0.3)
    fig.suptitle(scenario['title'])
    fig.tight_layout()
    plt.show()

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_3376\2551100443.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lakshya\AppData\Local\Temp\ipykernel_3376\2551100443.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lakshya\AppData\Local\Temp\ipykernel_3376\2551100443.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Lakshya\AppData\Local\Temp\ipykernel_3376\2551100443.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lakshya\AppData\Local\Temp\ipykernel_3376\2551100443.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Lakshya\AppData\Local\Temp\ipykernel_3376\2551100443.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Sanity checks and saved Phase 5 inputs

In [13]:
truth_table = classify_diagnostics(
    np.array([0.0, 2.0, 0.0, 2.0]),
    np.array([0.0, 0.0, 2.0, 2.0]),
    tau_r=1.0,
    tau_sigma=1.0,
)
assert np.array_equal(truth_table, np.arange(4))
assert thresholds['tau_r'] > 0 and thresholds['tau_sigma'] > 0
assert not model.training
assert all(
    np.isfinite(values[key]).all()
    for values in diagnostic_results.values()
    for key in ('mean', 'variance', 'residual')
)
assert all((values['variance'] >= 0).all() for values in diagnostic_results.values())
selected_sensitivity = next(
    row for row in sensitivity_rows
    if row['passes'] == FINAL_MC_PASSES and row['percentile'] == FINAL_PERCENTILE
)
assert selected_sensitivity['validation_false_alarm_rate'] <= 0.05

metrics_dir = project_root / 'results' / 'metrics'
configs_dir = project_root / 'results' / 'configs'
raw_dir = project_root / 'results' / 'raw'
metrics_dir.mkdir(parents=True, exist_ok=True)
configs_dir.mkdir(parents=True, exist_ok=True)
raw_dir.mkdir(parents=True, exist_ok=True)
reliability_config_path = configs_dir / 'reliability_config.json'
reliability_metrics_path = metrics_dir / 'reliability_metrics.json'
diagnostics_path = raw_dir / 'reliability_diagnostics.npz'

reliability_config = {
    'mc_passes': FINAL_MC_PASSES,
    'threshold_percentile': FINAL_PERCENTILE,
    **thresholds,
    'seed': SEED,
    'state_codes': {name: code for code, name in enumerate(DIAGNOSTIC_STATE_NAMES)},
    'validation_false_alarm_rate': selected_sensitivity['validation_false_alarm_rate'],
}
reliability_config_path.write_text(
    json.dumps(reliability_config, indent=2), encoding='utf-8'
)
reliability_metrics_path.write_text(
    json.dumps(
        {'scenarios': scenario_metrics, 'sensitivity': sensitivity_rows}, indent=2
    ),
    encoding='utf-8',
)

saved_arrays = {
    'scenario_names': np.array([scenario['name'] for scenario in scenarios]),
    'time': scenarios[0]['time'][window_length:].astype(np.float32),
    'tau_r': np.array(thresholds['tau_r']),
    'tau_sigma': np.array(thresholds['tau_sigma']),
}
for field in ('true', 'measured', 'voltage', 'load'):
    saved_arrays[field] = np.stack(
        [scenario[field][window_length:] for scenario in scenarios]
    ).astype(np.float32)
for field in ('mean', 'variance', 'residual', 'codes'):
    saved_arrays[field] = np.stack(
        [diagnostic_results[scenario['name']][field] for scenario in scenarios]
    )
np.savez_compressed(diagnostics_path, **saved_arrays)

with np.load(diagnostics_path) as saved:
    assert saved['codes'].shape == (len(scenarios), len(saved['time']))
print('All reliability sanity checks passed.')
print(f'Saved {reliability_config_path}, {reliability_metrics_path}, and {diagnostics_path}')

All reliability sanity checks passed.
Saved C:\Users\Lakshya\OneDrive\Desktop\cs\project\results\configs\reliability_config.json, C:\Users\Lakshya\OneDrive\Desktop\cs\project\results\metrics\reliability_metrics.json, and C:\Users\Lakshya\OneDrive\Desktop\cs\project\results\raw\reliability_diagnostics.npz


## Phase 4 conclusion

The sensitivity study found no material diagnostic gain beyond **N=10** MC passes: mean detection was 0.195 at N=10 and 0.200 at N=30 for the selected 99th-percentile calibration, while runtime rose from 5.8 s to 20.6 s. The initial 95th-percentile rule produced about **9.7%** combined clean-validation false alarms, so the final clean-only calibration uses the **99th percentile**, giving $\tau_r=0.7244$ rad/s and $\tau_\sigma=0.02396$ (rad/s)$^2$. Clean unseen operation then had a **1.35%** false-alarm rate.

The residual reliably detected **increased sensor noise** (88.8% combined detection), but it did **not** persistently detect a constant bias (2.6%) or dropout-to-zero (3.1%) because the autoregressive LSTM quickly accepted the faulty measurement history. MC Dropout did **not** detect the intended model mismatch: the outside-load and parameter-shift cases had only 1.9% and 1.1% combined detections and **0% meaningful MODEL_UNCERTAINTY/BOTH assignment** after the shift. Conversely, increased sensor noise often raised MC variance and was labeled MODEL_UNCERTAINTY or BOTH, so the two signals do not cleanly separate sensor faults from model mismatch. Bias/dropout adaptation, noisy-input uncertainty, and undetected plant shifts are the ambiguous/failing cases. These results are **not strong enough to use the proposed dual-reliability logic for adaptive MPC switching yet**; the trained LSTM remains credible, but Phase 5 should not assume these reliability signals are validated.